# Train New MTI Model from Scratch (3D)

This notebook trains a new **MTI-Net** model using `src/main3_infer.py` and verifies dataset readiness before training.

Reference alignment:
- Paper: *Iterative multitask learning and inference from seismic images* (Gao, 2024)
- Tasks: `RGT`, `DHR`, `fault probability`, `fault dip`, `fault strike`
- This notebook covers the **MTI** stage training from scratch.

In [ ]:
from pathlib import Path
import os
import re
import numpy as np
import subprocess

WORKSPACE = Path('/home/roderickperez/DataScienceProjects/multiTaskLearningSeismic')
TRAIN_ROOT = WORKSPACE / 'train' / 'dataset3'
DATA_TRAIN = TRAIN_ROOT / 'data_train'
TARGET_TRAIN = TRAIN_ROOT / 'target_train'
DATA_VALID = TRAIN_ROOT / 'data_valid'
TARGET_VALID = TRAIN_ROOT / 'target_valid'
OUT_DIR = WORKSPACE / 'result3_infer_new'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Workspace:', WORKSPACE)
print('Training data dir:', DATA_TRAIN)
print('Validation data dir:', DATA_VALID)
print('Output dir:', OUT_DIR)

## Step 1 — Audit dataset availability

Checks whether all expected files exist for each sample ID:
- input seismic: `<id>.bin`
- labels: `<id>_rgt.bin`, `<id>_dhr.bin`, `<id>_fsem.bin`, `<id>_fdip.bin`, `<id>_fstrike.bin`

In [ ]:
def sample_ids(data_dir: Path):
    ids = []
    for p in sorted(data_dir.glob('*.bin')):
        m = re.match(r'^(\d+)\.bin$', p.name)
        if m:
            ids.append(m.group(1))
    return ids

def audit_split(data_dir: Path, target_dir: Path, split_name: str):
    ids = sample_ids(data_dir)
    print(f'\n[{split_name}] ids={ids}')
    if not ids:
        print('  No samples found.')
        return ids

    for sid in ids:
        expected_data = [
            f'{sid}.bin',
            f'{sid}_rgt.bin', f'{sid}_dhr.bin', f'{sid}_fsem.bin',
            f'{sid}_fdip.bin', f'{sid}_fstrike.bin',
        ]
        expected_target = [
            f'{sid}_rgt.bin', f'{sid}_dhr.bin', f'{sid}_fsem.bin',
            f'{sid}_fdip.bin', f'{sid}_fstrike.bin',
        ]
        miss_data = [n for n in expected_data if not (data_dir / n).exists()]
        miss_target = [n for n in expected_target if not (target_dir / n).exists()]
        print(f'  id {sid}: missing data={miss_data or 
}, missing target={miss_target or 
}')
    return ids

train_ids = audit_split(DATA_TRAIN, TARGET_TRAIN, 'train')
valid_ids = audit_split(DATA_VALID, TARGET_VALID, 'valid')

print('\nSummary:')
print('  ntrain =', len(train_ids))
print('  nvalid =', len(valid_ids))

In [ ]:
def infer_cube_n(file_path: Path):
    n_elem = np.fromfile(file_path, dtype=np.float32).size
    n = int(round(n_elem ** (1/3)))
    return n_elem, n

for split_name, data_dir in [('train', DATA_TRAIN), ('valid', DATA_VALID)]:
    f = data_dir / '0.bin'
    if f.exists():
        n_elem, n = infer_cube_n(f)
        print(f'{split_name}: {f.name} elements={n_elem}, inferred cube size ~ {n}^3')
    else:
        print(f'{split_name}: sample 0.bin not found')

## Step 2 — Paper alignment notes

This repository’s dataset structure is aligned with MTI training targets in the paper:
- `RGT`: relative geological time
- `DHR`: denoised higher-resolution seismic
- `Fault`: probability (`fsem`), dip (`fdip`), strike (`fstrike`)

Important limitation in this workspace:
- The bundled 3D training example has only **1 train sample** and **1 validation sample** (`64^3`).
- This is enough to run training technically, but not enough for robust generalization on a full field volume.

## Step 3 — Configure training command

In [ ]:
# Match available demo cube size
N1 = 64
N2 = 64
N3 = 64
NTRAIN = max(1, len(train_ids))
NVALID = max(1, len(valid_ids))
EPOCHS = 100
BATCH_TRAIN = 1
BATCH_VALID = 1
GPUS = 1
LR = 0.5e-4

cmd = [
    'uv', 'run', 'python', 'src/main3_infer.py',
    f'--n1={N1}', f'--n2={N2}', f'--n3={N3}',
    f'--ntrain={NTRAIN}', f'--nvalid={NVALID}',
    f'--batch_train={BATCH_TRAIN}', f'--batch_valid={BATCH_VALID}',
    f'--epochs={EPOCHS}', f'--lr={LR}', f'--gpus_per_node={GPUS}',
    f'--dir_data_train={DATA_TRAIN}', f'--dir_data_valid={DATA_VALID}',
    f'--dir_target_train={TARGET_TRAIN}', f'--dir_target_valid={TARGET_VALID}',
    f'--dir_output={OUT_DIR}',
    '--rgt=y', '--dhr=y', '--fault=y'
]

print('Training command:')
print(' '.join(map(str, cmd)))

## Step 4 — Run training (optional)

Set `RUN_TRAINING=True` to start training from scratch.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    result = subprocess.run(cmd, cwd=str(WORKSPACE), check=False)
    print('Exit code:', result.returncode)
else:
    print('Training is disabled. Set RUN_TRAINING=True to execute.')

In [ ]:
ckpts = sorted(OUT_DIR.glob('*.ckpt'))
print('Checkpoint count:', len(ckpts))
for p in ckpts[-10:]:
    print(p.name)

if (OUT_DIR / 'last.ckpt').exists():
    print('\nReady for inference with:', OUT_DIR / 'last.ckpt')